In [6]:
%%capture
# 패키지 설치
%pip install "langchain>=0.3.0" "langchain-openai>=0.2.0" "langchain-upstage>=0.3.0" "langgraph>=0.2.0" "langchain-community>=0.3.0" "langchain-text-splitters>=0.3.0" "chromadb>=0.5.0" "tiktoken>=0.7.0" "python-dotenv>=1.0.0" -q

In [7]:
import os
import warnings
from dotenv import load_dotenv
warnings.filterwarnings("ignore")

# .env 파일에서 환경변수 로드
load_dotenv()

# API 키 확인 (.env 미설정 시 직접 입력)
# if not os.environ.get("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = input("🔑 OPENAI_API_KEY를 입력하세요: ")

if not os.environ.get("UPSTAGE_API_KEY"):
    os.environ["UPSTAGE_API_KEY"] = input("🔑 UPSTAGE_API_KEY를 입력하세요: ")
print("✅ 환경 설정 완료")

✅ 환경 설정 완료


In [8]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 모델 초기화 — OpenAI / Solar 중 택 1 (주석 해제하여 전환)

# ── Option A: OpenAI (기본) ──
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# embeddings = OpenAIEmbeddings()

# ── Option B: Solar 대안 (주석 해제하여 사용) ──
from langchain_upstage import ChatUpstage, UpstageEmbeddings
llm = ChatUpstage(model="solar-pro3")
embeddings = UpstageEmbeddings(model="embedding-query")

In [9]:
# LLM만으로 쿠폰 발급 요청
response = llm.invoke("주문 ORD001에 5000원 쿠폰을 발급해줘.")
print("LLM 응답:")
print(response.content)

LLM 응답:
주문 번호 **ORD001**에 대해 5,000원 쿠폰을 발급해드릴게요.

**쿠폰 정보**

- **쿠폰 ID:** C-20260330-ORD001  
- **금액:** 5,000원 할인  
- **유효 기간:** 발급일로부터 **90일** (예: 2026‑03‑30 → 2026‑06‑28까지 사용 가능)  
- **사용 제한:**  
  - 1회 사용 가능  
  - 다른 쿠폰과 중복 적용 불가  
  - 주문 금액이 5,000원 이상이어야 정상 할인 적용  

위 쿠폰을 사용하시려면 주문 시 결제 단계에서 **쿠폰 코드** `C-20260330-ORD001` 를 입력해 주세요.

추가로 도움이 필요하거나 다른 쿠폰·할인 관련 문의가 있으면 언제든 말씀해 주세요!


## Agent

### Tool을 만들어서 LLM이 행동 가능하게 만들기

In [10]:
# 정책 문서 생성 (실습용)
reward_policy = """
# 보상 정책 (Reward Policy)

## 쿠폰 발급 조건
1. 배송 지연 시: 최대 10,000원 쿠폰 발급 가능
2. 상품 불량 시: 최대 20,000원 쿠폰 발급 가능
3. 단순 변심: 쿠폰 발급 불가

## 쿠폰 발급 한도
- 1회 최대 발급 금액: 20,000원
- 월간 누적 한도: 50,000원

## 주의사항
- 배송 완료 후 7일 이내에만 보상 가능
- 이미 쿠폰을 받은 주문에는 추가 발급 불가
"""

shipping_policy = """
# 배송 정책 (Shipping Policy)

## 배송 상태
- 주문 접수: 주문이 접수된 상태
- 배송 준비중: 상품 포장 중
- 배송 중: 택배사에 인계됨
- 배송 완료: 고객에게 전달 완료
- 배송 지연: 예상 배송일을 초과함

## 배송 지연 기준
- 일반 배송: 결제 후 3영업일 초과 시 지연
- 새벽 배송: 주문 익일 미도착 시 지연
"""

# 파일로 저장
with open("reward_policy.txt", "w", encoding="utf-8") as f:
    f.write(reward_policy)

with open("shipping_policy.txt", "w", encoding="utf-8") as f:
    f.write(shipping_policy)

print("✅ 정책 문서 생성 완료")

✅ 정책 문서 생성 완료


In [11]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma

# 문서 로드
loaders = [
    TextLoader("reward_policy.txt", encoding="utf-8"),
    TextLoader("shipping_policy.txt", encoding="utf-8")
]

documents = []
for loader in loaders:
    documents.extend(loader.load())

# 텍스트 분할
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(documents)

vectorstore = Chroma.from_documents(split_docs, embeddings)

# Retriever 생성
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"✅ Vector Store 생성 완료 (문서 {len(split_docs)}개)")

✅ Vector Store 생성 완료 (문서 2개)


In [12]:
# TODO: RAG 정책 검색 도구 구현
# 아래 함수를 완성하세요.

def search_policy(query: str) -> str:
    """
    정책 문서를 검색하여 관련 내용을 반환합니다.

    Args:
        query: 검색할 질문 (예: "쿠폰 발급 한도")

    Returns:
        검색된 정책 내용
    """
    # TODO: 여기에 코드를 작성하세요.
    # 힌트: retriever.invoke(query)로 문서를 검색하고, 결과를 문자열로 합치세요.
    docs = retriever.invoke(query)
    if docs:
        return '\n\n'.join([doc.page_content for doc in docs])
    else:
        return '관련 정책을 찾을 수 없습니다.'

In [13]:
# 정책 검색 테스트
result = search_policy("쿠폰 발급 한도")
print("검색 결과:")
print(result)

검색 결과:
# 보상 정책 (Reward Policy)

## 쿠폰 발급 조건
1. 배송 지연 시: 최대 10,000원 쿠폰 발급 가능
2. 상품 불량 시: 최대 20,000원 쿠폰 발급 가능
3. 단순 변심: 쿠폰 발급 불가

## 쿠폰 발급 한도
- 1회 최대 발급 금액: 20,000원
- 월간 누적 한도: 50,000원

## 주의사항
- 배송 완료 후 7일 이내에만 보상 가능
- 이미 쿠폰을 받은 주문에는 추가 발급 불가

# 배송 정책 (Shipping Policy)

## 배송 상태
- 주문 접수: 주문이 접수된 상태
- 배송 준비중: 상품 포장 중
- 배송 중: 택배사에 인계됨
- 배송 완료: 고객에게 전달 완료
- 배송 지연: 예상 배송일을 초과함

## 배송 지연 기준
- 일반 배송: 결제 후 3영업일 초과 시 지연
- 새벽 배송: 주문 익일 미도착 시 지연


In [14]:
# 주문 데이터 (시뮬레이션용)
orders_db = {
    "ORD001": {"status": "배송 지연", "product": "노트북", "customer": "홍길동"},
    "ORD002": {"status": "배송 완료", "product": "키보드", "customer": "김철수"},
    "ORD003": {"status": "배송 중", "product": "마우스", "customer": "이영희"},
}

# 발급된 쿠폰 기록
issued_coupons = {}

def get_order_status(order_id: str) -> str:
    """주문 상태를 조회합니다."""
    if order_id in orders_db:
        order = orders_db[order_id]
        return f"주문번호: {order_id}, 상태: {order['status']}, 상품: {order['product']}"
    return f"주문 {order_id}을 찾을 수 없습니다."

def issue_coupon(order_id: str, amount: int) -> str:
    """
    쿠폰을 발급합니다. (검증 없는 기본 버전)
    주의: 이 함수는 검증 로직이 없어 정책 위반이 발생할 수 있습니다.
    """
    issued_coupons[order_id] = amount
    return f"✅ 주문 {order_id}에 {amount:,}원 쿠폰 발급 완료"

print("✅ 도구 함수 정의 완료")

✅ 도구 함수 정의 완료


In [15]:
from langchain_core.tools import tool

# tool 데코레이터를 사용하여 도구 정의
@tool
def search_policy_tool(query: str) -> str:
    """고객 서비스 정책(보상, 배송)을 검색합니다. 쿠폰 발급 조건, 한도 등을 확인할 때 사용합니다."""
    return search_policy(query)

@tool
def get_order_status_tool(order_id: str) -> str:
    """주문 상태를 조회합니다. 주문번호(예: ORD001)를 입력하면 현재 상태를 반환합니다."""
    return get_order_status(order_id)

@tool
def issue_coupon_tool(order_id: str, amount: int) -> str:
    """쿠폰을 발급합니다. 주문번호와 금액을 입력합니다."""
    return issue_coupon(order_id, amount)

# Tool 목록 정의
tools = [search_policy_tool, get_order_status_tool, issue_coupon_tool]

print("✅ Tool 목록 정의 완료")
for tool in tools:
    print(f"  - {tool.name}: {tool.description[:50]}...")

✅ Tool 목록 정의 완료
  - search_policy_tool: 고객 서비스 정책(보상, 배송)을 검색합니다. 쿠폰 발급 조건, 한도 등을 확인할 때 사용...
  - get_order_status_tool: 주문 상태를 조회합니다. 주문번호(예: ORD001)를 입력하면 현재 상태를 반환합니다....
  - issue_coupon_tool: 쿠폰을 발급합니다. 주문번호와 금액을 입력합니다....


In [16]:
from langgraph.prebuilt import create_react_agent

# 기본 Agent 프롬프트 (시스템 메시지)
system_prompt = "당신은 고객 서비스 AI 에이전트입니다. 고객의 요청을 처리하기 위해 제공된 도구를 사용하세요."

# Agent 생성 (LangGraph create_react_agent 사용)
agent_executor = create_react_agent(model=llm, tools=tools, prompt=system_prompt)

print("✅ Agent 생성 완료")

✅ Agent 생성 완료


In [20]:
# Agent 실행 테스트
response = agent_executor.invoke({
    "messages": [{"role": "user", "content": "주문 ORD001의 상태를 확인하고, 배송 지연이면 7500원 쿠폰을 발급해줘."}]
})
print("\n최종 응답:")
print(response["messages"][-1].content)


최종 응답:
주문 ORD001은 현재 **배송 지연** 상태입니다.  

요청하신 대로 7,500원 쿠폰을 발급해 드렸습니다. 쿠폰 번호는 아래와 같습니다.

**쿠폰 번호:** `COUPON-7500-ORD001`  

쿠폰을 이용해 다음 주문 시 혜택을 받으실 수 있습니다. 추가로 필요하신 사항이 있으면 언제든 알려 주세요!


In [21]:
issued_coupons

{'ORD001': 7500}

## 맥락 유지하기

In [23]:
# 첫 번째 대화
response1 = agent_executor.invoke({
    "messages": [{"role": "user", "content": "내 주문번호는 ORD002이야."}]
})
print("응답 1:", response1["messages"][-1].content)

# 두 번째 대화 - 이전 맥락을 기억하지 못함
response2 = agent_executor.invoke({
    "messages": [{"role": "user", "content": "아까 말한 주문의 상태가 뭐야?"}]
})
print("\n응답 2:", response2["messages"][-1].content)

응답 1: 안녕하세요! 주문번호 **ORD002**는 이미 **배송 완료** 상태이며, 상품은 **키보드**로 확인되었습니다.  

추가로 도와드릴 사항이 있으신가요?  
- 배송 관련 문의  
- 반품/교환 절차  
- 쿠폰 발급 여부 확인  
- 주문 내용 확인 등  

필요하신 것을 말씀해 주세요.

응답 2: 주문번호 **ORD001**은 현재 **“배송 지연”** 상태이며, 구매하신 상품은 **노트북**입니다.  

추가로 도움이 필요하시면 아래 정보를 알려주시면 됩니다.  

- 주문하신 날짜 또는 주문 확인 번호(예: 주문 ID)  
- 배송이 지연되는 구체적인 사유(예: 물류센터 내 재고 부족, 날씨 등)  
- 현재 보관 중인 배송 예정 일정 또는 예상 배송일  
- 다른 문의 사항(예: 교환·환불, 쿠폰·할인 적용 여부 등)  

필요한 부분을 알려주시면 상세히 안내해 드리겠습니다.


In [24]:
from langgraph.checkpoint.memory import MemorySaver

# Memory (체크포인터) 생성
memory = MemorySaver()

# Memory가 적용된 프롬프트
prompt_with_memory = """당신은 고객 서비스 AI 에이전트입니다. 대화 기록을 참고하여 응답하세요."""

# Memory가 적용된 Agent 생성
agent_with_memory = create_react_agent(
    model=llm,
    tools=tools,
    prompt=prompt_with_memory,
    checkpointer=memory  # 체크포인터로 대화 기록 유지
)

print("✅ Memory가 적용된 Agent 생성 완료")

✅ Memory가 적용된 Agent 생성 완료


In [25]:
# Memory 테스트
config = {"configurable": {"thread_id": "user123"}}

# 첫 번째 대화
response1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "내 주문번호는 ORD001이야."}]},
    config=config
)
print("응답 1:", response1["messages"][-1].content)

# 두 번째 대화 - 이제 맥락을 기억!
response2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "아까 말한 주문의 상태가 뭐야?"}]},
    config=config
)
print("\n응답 2:", response2["messages"][-1].content)

응답 1: 주문번호 **ORD001**에 대한 현재 상황을 확인해 드렸습니다.

- **상품**: 노트북  
- **주문 상태**: **배송 지연**

현재 배송이 지연되고 있어 불편을 드려 죄송합니다.  
배송 지연에 대해 보상 쿠폰이 필요하시다면, 제공해 드릴 수 있습니다. 아래 정보를 알려 주시면 바로 쿠폰을 발급해 드리겠습니다.

1. **쿠폰 금액** (예: 5,000원, 10,000원 등)  
2. **쿠폰을 사용하실 이메일 주소** (또는 계정 ID)

추가로 궁금하신 점이나 다른 요청이 있으시면 편하게 말씀해 주세요!

응답 2: 주문번호 **ORD001**의 현재 상태는 **배송 지연**입니다.  
해당 주문은 **노트북** 상품이며, 아직 배송이 시작되지 않아 지연되고 있는 상황입니다.  

추가로 보상 쿠폰을 발급해 드릴 수 있도록, 다음 정보를 알려 주시면 바로 처리해 드리겠습니다.

1. **쿠폰 금액** (예: 5,000원, 10,000원 등)  
2. **쿠폰을 받으실 이메일 주소** (또는 계정 ID)

필요하시면 말씀해 주세요!


## ReAct

### create react agent

In [26]:
# 복잡한 요청
complex_request = """
주문 ORD001, ORD002, ORD003의 상태를 모두 확인하고,
배송 지연인 주문에만 쿠폰을 발급해줘.
"""

response = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": complex_request}]},
    config=config
)
print("응답:", response["messages"][-1].content)

응답: 현재 주문 3건의 상태는 다음과 같습니다:

| 주문번호 | 상품   | 현재 상태 |
|----------|--------|-----------|
| **ORD001** | 노트북 | **배송 지연** |
| **ORD002** | 키보드 | 배송 완료 |
| **ORD003** | 마우스 | 배송 중 |

**배송이 지연되고 있는 주문(ORD001)​에 대해 1,000원 쿠폰을 발급해 드렸습니다.**  
쿠폰은 귀하의 계정(이메일: **user@example.com**)으로 바로 전송되었습니다.

추가로 다른 쿠폰 금액을 원하시거나, 다른 도움이 필요하시면 언제든 알려 주세요!


### StateGraph

In [28]:
# =============================================================================
# StateGraph 공통 컴포넌트 정의
# - llm: Step 2에서 정의한 LLM 인스턴스
# - tools: Step 3에서 정의한 도구 목록
# =============================================================================

from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage

# AgentState 정의: MessagesState를 상속하여 messages 필드를 자동으로 사용
# - MessagesState에 messages 필드가 내장되어 있어 별도 정의가 불필요합니다.
# - 필요시 커스텀 필드를 추가하여 확장할 수 있습니다.
class AgentState(MessagesState):
    """Agent의 상태를 정의합니다."""
    pass

print("✅ AgentState 정의 완료")
print("   - MessagesState를 상속하여 messages 필드가 자동으로 포함됩니다.")

✅ AgentState 정의 완료
   - MessagesState를 상속하여 messages 필드가 자동으로 포함됩니다.


In [29]:
# 노드 함수 정의: Graph에 등록할 함수들

def call_agent(state: AgentState) -> AgentState:
    """LLM을 호출하여 다음 행동을 결정합니다."""
    # LLM에 도구를 바인딩하여 호출
    response = llm.bind_tools(tools).invoke(state["messages"])
    return {"messages": [response]}

def call_tools(state: AgentState) -> AgentState:
    """도구를 실행합니다."""
    last_message = state["messages"][-1]
    tool_messages = []
    
    # 각 도구 호출에 대해 실행
    for tool_call in last_message.tool_calls:
        for t in tools:
            if t.name == tool_call["name"]:
                result = t.invoke(tool_call["args"])
                tool_messages.append(ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                ))
                break
    
    return {"messages": tool_messages}

print("✅ 노드 함수 정의 완료")
print("   - call_agent: LLM을 호출하여 응답 또는 도구 호출 결정")
print("   - call_tools: 선택된 도구를 실제로 실행")

✅ 노드 함수 정의 완료
   - call_agent: LLM을 호출하여 응답 또는 도구 호출 결정
   - call_tools: 선택된 도구를 실제로 실행


In [30]:
# TODO: Direct 패턴 StateGraph 구현
from langgraph.graph import StateGraph, START, END

# TODO: 여기에 코드를 작성하세요.
# 힌트: StateGraph를 구성하세요
#   - workflow_direct = StateGraph(AgentState)
#   - add_node로 "agent", "tools" 노드 추가
#   - add_edge(START, "agent")
#   - add_edge("agent", "tools")
#   - add_edge("tools", END)
#   - workflow_direct.compile()
# StateGraph 생성
workflow_direct = StateGraph(AgentState)

# 노드 추가
workflow_direct.add_node('agent', call_agent)
workflow_direct.add_node('tools', call_tools)

# 간선 추가
workflow_direct.add_edge(START, 'agent')
workflow_direct.add_edge('agent', 'tools')
workflow_direct.add_edge('tools', END)

# 컴파일
agent_direct = workflow_direct.compile()

print("✅ Direct 패턴 Agent 생성 완료")

✅ Direct 패턴 Agent 생성 완료


In [31]:
# Direct 패턴 테스트: 단순 요청 (1회 도구 호출)
print("=== Direct 패턴 테스트 ===")

simple_request = "주문 ORD001의 상태를 확인해줘."

initial_state = {"messages": [HumanMessage(content=simple_request)]}
result = agent_direct.invoke(initial_state)

# 도구 호출 횟수 확인
tool_call_count = sum(1 for msg in result["messages"] if isinstance(msg, ToolMessage))
print(f"도구 호출 횟수: {tool_call_count}회 (Direct 패턴은 최대 1회)")

# 메시지 흐름 출력
print("\n메시지 흐름:")
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    content_preview = msg.content[:100] if msg.content else "(도구 호출)"
    print(f"  [{i+1}] {msg_type}: {content_preview}...")

=== Direct 패턴 테스트 ===
도구 호출 횟수: 1회 (Direct 패턴은 최대 1회)

메시지 흐름:
  [1] HumanMessage: 주문 ORD001의 상태를 확인해줘....
  [2] AIMessage: (도구 호출)...
  [3] ToolMessage: 주문번호: ORD001, 상태: 배송 지연, 상품: 노트북...


In [32]:
# ReAct 시스템 프롬프트 (제공 코드)
# Thought → Action → Observation 순환을 유도하는 프롬프트

react_prompt_stategraph = """당신은 고객 서비스 AI 에이전트입니다.

다음의 ReAct 프레임워크를 따라 요청을 처리하세요:

1. Thought: 현재 상황을 분석하고 다음에 할 일을 생각합니다.
2. Action: 필요한 도구를 실행합니다.
3. Observation: 도구 실행 결과를 관찰합니다.
4. 위 과정을 반복하여 작업을 완료합니다.
5. Final Answer: 모든 작업이 완료되면 최종 응답을 제공합니다.

중요 원칙:
- 각 단계에서 Observation 결과를 반드시 확인하세요.
- 이전 결과를 바탕으로 다음 행동을 결정하세요.
- 정책 확인 → 주문 상태 확인 → 조건 판단 → 실행 순서를 지키세요."""

print("✅ ReAct 시스템 프롬프트 정의 완료")

✅ ReAct 시스템 프롬프트 정의 완료


In [36]:
# TODO: ReAct 분기 함수 + 순환 엣지 구현
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, START, END

# call_agent_react 함수 (제공 코드)
def call_agent_react(state: AgentState) -> AgentState:
    """ReAct용 LLM 호출 (시스템 프롬프트 포함)"""
    messages = state["messages"]
    
    # 시스템 프롬프트가 없으면 추가
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=react_prompt_stategraph)] + list(messages)
    
    response = llm.bind_tools(tools).invoke(messages)
    return {"messages": [response]}

# TODO: 여기에 코드를 작성하세요.
# 힌트 1: should_continue_react 함수를 정의하세요
#   - 마지막 메시지에 tool_calls가 있으면 "tools" 반환
#   - 없으면 "end" 반환
def should_continue_react(state: AgentState) -> str:
    """ReAct: 도구 호출이 있으면 계속, 없으면 종료"""
    last_message = state['messages'][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return 'tools'
    return 'end'

# 힌트 2: StateGraph를 구성하세요
#   - workflow_react = StateGraph(AgentState)
#   - add_node로 "agent", "tools" 노드 추가
#   - add_edge(START, "agent")
#   - add_conditional_edges로 조건부 분기
#   - ★ add_edge("tools", "agent")로 ReAct 순환 구조 완성
#   - workflow_react.compile()
workflow_react = StateGraph(AgentState)

# Direct와 사실상 동일
workflow_react.add_node('agent', call_agent_react)
workflow_react.add_node('tools', call_tools)
workflow_react.add_edge(START, 'agent')

# 조건부 엣지
workflow_react.add_conditional_edges(
    'agent',
    should_continue_react,
    # should_continue_react의 결과에 따라
    # tools였으면 tools 노드로
    # end였으면 end 노드로
    {'tools': 'tools', 'end': END}
)

# tools 호출 후 다시 agent로
workflow_react.add_edge('tools', 'agent')

# 컴파일
agent_react_stategraph = workflow_react.compile()

print("✅ ReAct 패턴 Agent 생성 완료")

✅ ReAct 패턴 Agent 생성 완료


In [37]:
# ReAct 패턴 그래프 시각화
print("=== ReAct 패턴 그래프 구조 ===")
print(agent_react_stategraph.get_graph().draw_mermaid())

print("\n=== Direct vs ReAct 비교 ===")
print("""
Direct:  [agent] → [tools] → [END]     (1회 실행)
ReAct:   [agent] → [tools] → [agent]   (반복 가능)
""")

=== ReAct 패턴 그래프 구조 ===
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -. &nbsp;end&nbsp; .-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


=== Direct vs ReAct 비교 ===

Direct:  [agent] → [tools] → [END]     (1회 실행)
ReAct:   [agent] → [tools] → [agent]   (반복 가능)



In [38]:
# ReAct 패턴 테스트: 복잡한 요청 (여러 단계 필요)
print("=== ReAct 패턴 테스트 ===")

complex_request = """
주문 ORD001, ORD002의 상태를 확인하고,
배송 지연인 주문에만 정책에 맞는 금액의 쿠폰을 발급해줘.
"""

initial_state = {"messages": [HumanMessage(content=complex_request)]}
result = agent_react_stategraph.invoke(initial_state)

# 도구 호출 횟수 확인
tool_call_count = sum(1 for msg in result["messages"] if isinstance(msg, ToolMessage))
print(f"도구 호출 횟수: {tool_call_count}회 (ReAct 패턴은 필요한 만큼 반복)")

# 최종 응답
print("\n최종 응답:")
print(result["messages"][-1].content)

=== ReAct 패턴 테스트 ===
도구 호출 횟수: 4회 (ReAct 패턴은 필요한 만큼 반복)

최종 응답:
주문 상태를 확인했으며, **ORD001**만 배송 지연 상태이므로 정책에 따라 **10,000원** 쿠폰을 발급했습니다.  

- **ORD001**: 배송 지연 → 10,000원 쿠폰 발급 완료  
- **ORD002**: 배송 완료 → 쿠폰 발급 대상이 아님  

추가 요청사항이 있으면 알려 주세요!


In [39]:
issued_coupons

{'ORD001': 10000}